## Changelog
- parent: 20260503_230404_f1e5a8ee
- change: refactor preprocessing+model into sklearn Pipeline; CV now uses
  KFold(5, shuffle=True, random_state=42) and refits imputers/encoders per fold
- hypothesis: removing preprocessing leakage gives a more honest CV estimate;
  shuffled folds reduce CV noise. Public LB should not change materially since
  the model and feature set are unchanged — this is a measurement fix, not
  a modeling change.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "eda").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from src.eda import (
    numeric_columns,
    plot_histograms,
    plot_kde,
    plot_boxplots,
    plot_distributions,
    describe_df,
    get_missing_values,
    count_duplicates,
    plot_categorical_vs_target,
    plot_numerical_vs_target,
    target_rate_table,
    grouped_median,
)

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")


## Model

  A single cell that does three things: prepare data, evaluate via cross-validation, fit on the full training set.                      
   
  1. X / y preparation                                                                                                                  
                                                         
  X = train_data_raw.drop(columns=["SalePrice"])
  y = np.log1p(train_data_raw["SalePrice"])
                                                                                                                                        
  Features are the raw training set without the target. The log1p transform on the target is driven by the competition metric (RMSE on  
  log(price)): training on the log scale means the loss the model minimizes coincides with the metric being scored, and proportional    
  errors count the same on a 100k house as on a 500k house.                                                                             
                                                         
  2. Building the Pipeline

  pipe = build_pipeline(
      RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)
  )                                                                                                                                     
   
  build_pipeline (in utils/ames_sklearn_pipeline.py) returns a sklearn.pipeline.Pipeline with four steps:                               
                                                         
  1. AmesNAImputer — learned from train: LotFrontage median per Neighborhood, mode of Electrical/MSZoning/KitchenQual/etc.,             
  quality-ladder recoding where NA means "feature absent".
  2. add_temporal_features (stateless FunctionTransformer) — derives HouseAge, YearsSinceRemod, WasRemodeled, IsNewHouse; drops         
  YearBuilt/YearRemodAdd/YrSold.                                                                                                        
  3. AmesEncoder — quality columns (Ex/Gd/TA/Fa/Po → 5..1), CentralAir → 0/1, one-hot for the rest, fitted on the training fold.
  4. RandomForestRegressor — 300 trees, parallel, fixed seed.                                                                           
                                                         
  The key point: steps 1 and 3 are stateful (they learn statistics from train). By living inside the Pipeline, sklearn refits them on   
  every CV fold. This is what removes the leakage the previous code had.
                                                                                                                                        
  3. Cross-validation                                    

  cv = KFold(n_splits=5, shuffle=True, random_state=42)
  scores = cross_val_score(                                                                                                             
      pipe, X, y,
      cv=cv,                                                                                                                            
      scoring="neg_root_mean_squared_error",             
      n_jobs=-1,                                                                                                                        
  )
                                                                                                                                        
  Honest generalization-error estimate:                  
  - 5 folds with shuffle=True → i.i.d. folds, immune to ordering by Id.
  - random_state=42 → CV reproducible run after run.                                                                                    
  - neg_root_mean_squared_error against log-transformed y → the printed number is RMSE on log(price) directly, same unit as the Kaggle 
  metric.                                                                                                                               
  - n_jobs=-1 parallelizes the 5 folds; combined with the RF's own n_jobs=-1 it saturates all cores.                                    
                                                                                                    
  print(f"CV RMSE (log-price): {rmse.mean():.4f} ± {rmse.std():.4f}")                                                                   
  print(f"Per-fold:            {np.round(rmse, 4).tolist()}")                                                                           
                                                                                                                                        
  Output: mean ± std (the headline) plus the per-fold list (to spot an outlier fold). For this run: 0.1431 ± 0.0187, folds [0.146,      
  0.126, 0.175, 0.147, 0.122].                                                                                                          
                                                                                                                                        
  4. Final fit on the full training set                                                                                                 
  
  pipe.fit(X, y)                                                                                                                        
                                                         
  cross_val_score only estimates performance — it does not leave a fitted model behind. To produce test predictions you need a model    
  trained on all available training data. After this line pipe is ready for pipe.predict(test_data_raw).
                                                                                                                                        
  Why this structure                                     

  Three properties the previous code didn't have:                                                                                       
  - No leakage: AmesNAImputer.fit on a fold sees only that fold; LotFrontage neighborhood median, MSZoning mode, the one-hot vocabulary
  — all computed without looking at the validation portion.                                                                             
  - Reproducibility: two seeds (KFold + RF) → same number every time.
  - Model swap-friendly: build_pipeline accepts any regressor. The next experiment changes one line — RandomForestRegressor(...) → e.g. 
  GradientBoostingRegressor(...) — and preprocessing stays identical, making the comparison apples-to-apples.  

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold

from utils.ames_sklearn_pipeline import build_pipeline

X = train_data_raw.drop(columns=["SalePrice"])
y = np.log1p(train_data_raw["SalePrice"])

pipe = build_pipeline(
    RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X, y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rmse = -scores
print(f"CV RMSE (log-price): {rmse.mean():.4f} ± {rmse.std():.4f}")
print(f"Per-fold:            {np.round(rmse, 4).tolist()}")

pipe.fit(X, y)

In [ ]:
test_pred = np.expm1(pipe.predict(test_data_raw))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)